# SignSight — ViTSD ZIP: audit và kế hoạch làm sạch CAR35

Chọn **CPU runtime**. Notebook mount Drive của bạn, không dùng Kaggle/gdown, không cần GPU hay YOLO.
Bạn chỉ cần sửa `DRIVE_ZIP_PATH` thành đường dẫn file ZIP đang có trên Drive.

Mặc định chép ZIP **một lần** sang `/content`, đọc ảnh/nhãn trực tiếp trong ZIP, không giải nén toàn bộ.
File gốc trên Drive được giữ nguyên. Có thanh tiến độ copy/audit/dHash và cache audit trong cùng runtime.
Lưu notebook không giữ `/content`; báo cáo ZIP được lưu vào Drive và có thể tải về máy.

**Đầu ra là báo cáo trước train**: lỗi nhãn, ảnh trùng, ứng viên gần trùng, bảng mapping CAR35 và ảnh minh họa.
Chưa tự sửa nhãn, xoá ảnh, chia lại train/val/test hoặc train. Mapping tên giống nhau vẫn phải duyệt bằng mắt.


In [ ]:
import sys, subprocess
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q',
    'Pillow>=10,<13', 'PyYAML>=6,<7', 'tqdm>=4.66,<5', 'matplotlib>=3.8,<4'])
print('Dùng CPU runtime; không cần cài Torch/Ultralytics.')


In [ ]:
from google.colab import drive
drive.mount('/content/drive')


## 1. Chọn file ZIP trên Drive

Trong thanh Files bên trái Colab: mở `drive/MyDrive`, tìm file ZIP → menu ba chấm → **Copy path**,
rồi dán vào `DRIVE_ZIP_PATH`. Ví dụ `/content/drive/MyDrive/datasets/ViTSD.zip`.
Không dán link chia sẻ vào biến đường dẫn. Việc mount cần bạn xác nhận truy cập tài khoản Google.

Giữ `MAX_IMAGES=None` để audit toàn bộ. Có thể dùng 200 để thử đường dẫn trước; báo cáo sẽ ghi rõ partial.
`ANNOTATION_FORMAT='yolo'` đọc TXT, bỏ COCO cùng nguồn để không đếm đôi. Chọn `'coco'` nếu cần khảo sát bản JSON riêng.
`NEAR_HASH_DISTANCE=4` thêm bước đọc lại ảnh ZIP local để tìm liên kết dHash; có thể đặt `None` để bỏ bước này.

Tham chiếu CAR35 là **tuỳ chọn**: `REFERENCE_MANIFEST_PATH` nhận `manifest.json` V4 đã lưu trên Drive
hoặc ZIP có đúng một manifest. Chỉ cần manifest, không cần upload lại ảnh CAR35.
Nếu bỏ trống, notebook vẫn chạy; báo cáo ghi **chưa kiểm tra trùng xuyên nguồn**. Tham chiếu hash không kiểm tra cảnh gần giống xuyên nguồn.


In [ ]:
from pathlib import Path
import sys, json, shutil, hashlib
from datetime import datetime, timezone
WORK = Path('/content/signsight_vitsd_review')
WORK.mkdir(parents=True, exist_ok=True)
DRIVE_ZIP_PATH = ''  # Ví dụ /content/drive/MyDrive/datasets/ViTSD.zip
REFERENCE_MANIFEST_PATH = ''  # Tuỳ chọn: manifest.json CAR35 V4 hoặc ZIP chứa manifest
COPY_ZIP_TO_LOCAL = True
ANNOTATION_FORMAT = 'yolo'
MAX_IMAGES = None  # Toàn bộ; dùng 200 chỉ để smoke test, không phải full audit
NEAR_HASH_DISTANCE = 4  # None: bỏ dHash; không tự thay đổi split
SAVE_REPORT_TO_DRIVE = True
DRIVE_REPORT_DIR = Path('/content/drive/MyDrive/SignSight_Reports')
DOWNLOAD_REPORT = True
print('Workspace:', WORK)
print(f'Local free: {shutil.disk_usage(WORK).free/1024**3:.1f} GiB')


In [ ]:
# Helpers được nhúng để notebook upload chạy độc lập, không clone repo.
import importlib, sys, hashlib, json
TRAINING_FILES = {'training/__init__.py': '"""Offline datasets, experiments and notebook tooling; never imported by the API."""\n', 'training/data/__init__.py': '', 'training/data/common.py': "import hashlib, json\nfrom pathlib import Path\n\nCAR35 = [\n    'Speed_limit_40', 'Speed_limit_50', 'Speed_limit_60', 'Speed_limit_70', 'Speed_limit_80',\n    'No_entry', 'No_cars', 'No_cars_turn_left', 'No_cars_turn_right',\n    'No_turn_left', 'No_turn_right', 'No_U_turn', 'No_cars_U_turn',\n    'No_straight_ahead', 'No_turn_left_and_U_turn', 'No_turn_left_and_right',\n    'No_parking', 'No_stopping_and_parking', 'No_parking_odd_days', 'No_parking_even_days',\n    'Go_straight', 'Keep_right', 'Roundabout', 'One_way_road', 'Road_reserved_for_cars',\n    'Pedestrian_crossing', 'Children_crossing', 'Road_works', 'Slow_down',\n    'Dangerous_curve_left', 'Dangerous_curve_right',\n    'No_overtaking', 'End_no_overtaking', 'Give_way', 'End_all_prohibitions',\n]\nSPLITS = ('train', 'val', 'test')\nVERSION = 2\n\n\ndef digest_file(path):\n    with Path(path).open('rb') as stream:\n        digest = hashlib.sha256()\n        for block in iter(lambda: stream.read(1024*1024), b''): digest.update(block)\n        return digest.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')\n\n\ndef read_json(path):\n    return json.loads(Path(path).read_text(encoding='utf-8'))\n\n\ndef recipe(classes=CAR35, seed=42, block_size=100, near_threshold=4, groups=None):\n    from training.compatibility import implementation_digest\n    group_digest = hashlib.sha256(json.dumps(groups, sort_keys=True).encode()).hexdigest()\n    return dict(version=VERSION, classes=list(classes), seed=seed, block_size=block_size,\n        near_threshold=near_threshold, groups_sha256=group_digest,\n        implementation_sha256=implementation_digest(),\n        reader_sha256=hashlib.sha256(Path(__file__).with_name('audit.py').read_text(encoding='utf-8').encode()).hexdigest())\n", 'training/data/audit.py': '"""Read-only detection EDA, including files inside ZIPs. No training or remapping."""\nfrom collections import Counter, defaultdict\nimport hashlib\nimport json\nimport math\nfrom pathlib import Path, PurePosixPath\nimport random\nimport xml.etree.ElementTree as ET\nimport zipfile\n\nfrom PIL import Image\nimport yaml\n\nIMAGE_EXTENSIONS = {\'.jpg\', \'.jpeg\', \'.png\', \'.bmp\', \'.webp\', \'.tif\', \'.tiff\'}\n\n\nclass DatasetReader:\n    def __init__(self, root):\n        self.root = Path(root).resolve()\n        if self.root.is_file() and self.root.suffix.lower() == \'.zip\':\n            paths, base = [self.root], self.root.parent\n        elif self.root.is_dir():\n            paths, base = sorted(self.root.rglob(\'*\')), self.root\n        else:\n            raise FileNotFoundError(f\'Dataset directory or ZIP not found: {self.root}\')\n        self.entries, self.zips, self.warnings = {}, {}, []\n        for path in paths:\n            if not path.is_file() or path.is_symlink():\n                continue\n            name = path.relative_to(base).as_posix()\n            self.entries[name] = (path, None, path.stat().st_size)\n            if path.suffix.lower() == \'.zip\':\n                try:\n                    archive = zipfile.ZipFile(path)\n                    self.zips[name] = archive\n                    for item in archive.infolist():\n                        if not item.is_dir():\n                            self.entries[name + \'!/\' + item.filename] = (archive, item, item.file_size)\n                            if PurePosixPath(item.filename).suffix.lower() in {\'.zip\', \'.rar\', \'.7z\', \'.gz\', \'.tar\'}:\n                                self.warnings.append(f\'Nested archive not expanded: {name}!/{item.filename}\')\n                except (OSError, zipfile.BadZipFile) as error:\n                    self.warnings.append(f\'Cannot open archive {name}: {type(error).__name__}\')\n\n    def open(self, name):\n        source, member, _ = self.entries[name]\n        return source.open(member) if member is not None else source.open(\'rb\')\n\n    def read(self, name, max_bytes=256 * 1024 * 1024):\n        if self.entries[name][2] > max_bytes:\n            raise ValueError(f\'Annotation exceeds {max_bytes // 1024**2} MiB read limit\')\n        with self.open(name) as stream:\n            value = stream.read(max_bytes + 1)\n        if len(value) > max_bytes:\n            raise ValueError(\'Annotation read limit exceeded\')\n        return value\n\n    def close(self):\n        for archive in self.zips.values():\n            archive.close()\n\n    def __enter__(self):\n        return self\n\n    def __exit__(self, *_):\n        self.close()\n\n\ndef split_name(name):\n    mapping = {\'train\': \'train\', \'training\': \'train\', \'val\': \'val\', \'valid\': \'val\',\n               \'validation\': \'val\', \'test\': \'test\', \'testing\': \'test\'}\n    found = {mapping[part.casefold()] for part in PurePosixPath(name).parts[:-1] if part.casefold() in mapping}\n    return next(iter(found)) if len(found) == 1 else \'unknown\'\n\n\ndef paired_files(name, extension, directory_pairs, case_names):\n    path = PurePosixPath(name)\n    candidates = {str(path.with_suffix(extension))}\n    parts = list(path.parts)\n    for i, part in enumerate(parts[:-1]):\n        for before, after in directory_pairs:\n            if part.casefold() == before.casefold():\n                changed = parts.copy(); changed[i] = after\n                candidates.add(str(PurePosixPath(*changed).with_suffix(extension)))\n    return sorted({actual for candidate in candidates for actual in case_names.get(candidate.casefold(), [])})\n\n\ndef audit_dataset(reader, dataset, image_limit=None, hash_limit=2000, seed=42,\n                  verify_decode=False, progress=None, annotation_format=\'auto\'):\n    """All counts refer to scanned images; partial/hash coverage is explicit.\n\n    Zero objects means empty_annotation only for a recognized annotation file.\n    Missing/unreadable/ambiguous labels never become verified background images.\n    """\n    if annotation_format not in {\'auto\', \'yolo\', \'voc\', \'coco\'}:\n        raise ValueError(\'annotation format must be auto, yolo, voc or coco\')\n    rng = random.Random(seed)\n    images = sorted(name for name in reader.entries if PurePosixPath(name).suffix.lower() in IMAGE_EXTENSIONS)\n    if (image_limit is not None and image_limit < 0) or (hash_limit is not None and hash_limit < 0):\n        raise ValueError(\'Limits must be nonnegative or None\')\n    selected = images if image_limit is None else sorted(rng.sample(images, min(image_limit, len(images))))\n    hashed = set(selected if hash_limit is None else rng.sample(selected, min(hash_limit, len(selected))))\n    issues, image_rows, boxes, catalogs = [], [], [], []\n    case_names = defaultdict(list)\n    for name in reader.entries:\n        case_names[name.casefold()].append(name)\n\n    def issue(path, message):\n        issues.append({\'dataset\': dataset, \'path\': path, \'message\': str(message)[:600]})\n\n    for warning in reader.warnings:\n        issue(\'\', warning)\n    for name in reader.entries:\n        suffix = PurePosixPath(name).suffix.lower()\n        try:\n            names = None\n            if suffix in {\'.yaml\', \'.yml\'}:\n                config = yaml.safe_load(reader.read(name, 1024**2))\n                if isinstance(config, dict) and \'names\' in config:\n                    names = config[\'names\']\n            elif PurePosixPath(name).name.casefold() in {\'classes.txt\', \'obj.names\', \'classes.names\'}:\n                names = reader.read(name, 1024**2).decode(\'utf-8-sig\').splitlines()\n            if names is not None:\n                if isinstance(names, list):\n                    names = dict(enumerate(names))\n                if not isinstance(names, dict) or not names:\n                    raise ValueError(\'Invalid class names\')\n                normalized = {int(key): str(value).strip() for key, value in names.items()}\n                if any(key < 0 or not value for key, value in normalized.items()):\n                    raise ValueError(\'Invalid class ID/name\')\n                catalogs.append((name, normalized))\n        except Exception as error:\n            issue(name, f\'Class metadata: {type(error).__name__}: {error}\')\n\n    def catalog_for(image_name):\n        matches = [(name, names) for name, names in catalogs\n                   if str(PurePosixPath(name).parent) == \'.\'\n                   or image_name.startswith(str(PurePosixPath(name).parent) + \'/\')]\n        if matches:\n            depth = max(len(PurePosixPath(name).parent.parts) for name, _ in matches)\n            closest = [(name, names) for name, names in matches if len(PurePosixPath(name).parent.parts) == depth]\n            if len({json.dumps(names, sort_keys=True) for _, names in closest}) > 1:\n                issue(image_name, \'Conflicting class maps at the same directory; class IDs remain unmapped.\')\n                return (\'ambiguous-class-map\', {})\n            return closest[0]\n        unique = {json.dumps(names, sort_keys=True) for _, names in catalogs}\n        return catalogs[0] if len(unique) == 1 else (\'unmapped\', {})\n\n    # COCO indexing: resolve full/suffix names; ambiguous duplicate basenames are errors.\n    by_basename = defaultdict(list)\n    for name in images:\n        by_basename[PurePosixPath(name).name].append(name)\n    coco = defaultdict(list)\n    for name in reader.entries:\n        if annotation_format not in {\'auto\', \'coco\'} or PurePosixPath(name).suffix.lower() != \'.json\':\n            continue\n        try:\n            content = json.loads(reader.read(name))\n            if not isinstance(content, dict) or not {\'images\', \'annotations\', \'categories\'} <= content.keys():\n                issue(name, \'JSON is not COCO detection; inspect separately (e.g. LabelMe/classification).\')\n                continue\n            categories = {row[\'id\']: str(row[\'name\']) for row in content[\'categories\']}\n            annotations = defaultdict(list)\n            for annotation in content[\'annotations\']:\n                annotations[annotation[\'image_id\']].append(annotation)\n            for image in content[\'images\']:\n                filename = str(image[\'file_name\']).replace(\'\\\\\', \'/\')\n                candidates = by_basename[PurePosixPath(filename).name]\n                suffix_matches = [key for key in candidates if key == filename or key.endswith(\'/\' + filename)]\n                if suffix_matches:\n                    candidates = suffix_matches\n                # Do not join a JSON in one archive to same-named images in another.\n                container = name.split(\'!/\', 1)[0] if \'!/\' in name else None\n                candidates = [key for key in candidates if\n                              (key.startswith(container + \'!/\') if container else \'!/\' not in key)]\n                if len(candidates) != 1:\n                    issue(name, f\'COCO image reference missing/ambiguous: {filename}\')\n                    continue\n                coco[candidates[0]].append((name, categories, image, annotations[image[\'id\']]))\n        except Exception as error:\n            issue(name, f\'JSON: {type(error).__name__}: {error}\')\n\n    iterator = progress(selected, desc=dataset, unit=\'image\') if progress else selected\n    for name in iterator:\n        row = {\'dataset\': dataset, \'path\': name, \'split\': split_name(name), \'parent_folder\': str(PurePosixPath(name).parent),\n               \'width\': None, \'height\': None, \'mode\': None, \'format\': None,\n               \'annotation_status\': \'missing_annotation\', \'boxes\': 0, \'sha256\': None}\n        image_rows.append(row)\n        try:\n            with reader.open(name) as stream, Image.open(stream) as image:\n                row.update(width=image.width, height=image.height, mode=image.mode)\n                if verify_decode:\n                    image.load()\n            if name in hashed:\n                digest = hashlib.sha256()\n                with reader.open(name) as stream:\n                    for block in iter(lambda: stream.read(1024 * 1024), b\'\'):\n                        digest.update(block)\n                row[\'sha256\'] = digest.hexdigest()\n        except Exception as error:\n            row[\'annotation_status\'] = \'unreadable_image\'\n            issue(name, f\'Image: {type(error).__name__}: {error}\')\n            continue\n        txt = paired_files(name, \'.txt\', [(\'images\', \'labels\')], case_names)\n        xml = paired_files(name, \'.xml\', [(\'JPEGImages\', \'Annotations\'), (\'images\', \'annotations\')], case_names)\n        options = [(\'yolo\', path) for path in txt] + [(\'voc\', path) for path in xml] + [(\'coco\', entry) for entry in coco[name]]\n        if annotation_format != \'auto\':\n            options = [option for option in options if option[0] == annotation_format]\n        if not options:\n            continue\n        if len(options) > 1:\n            row[\'annotation_status\'] = \'ambiguous_annotation\'\n            issue(name, \'Multiple matching annotations/formats; select a single export/root before training.\')\n            continue\n        kind, annotation = options[0]\n        row[\'format\'] = kind\n        raw_boxes, bad = [], False\n        try:\n            if kind == \'yolo\':\n                source, names = catalog_for(name)\n                for line_number, line in enumerate(reader.read(annotation, 8 * 1024**2).decode(\'utf-8-sig\').splitlines(), 1):\n                    if not line.strip():\n                        continue\n                    try:\n                        fields = line.split()\n                        if len(fields) != 5:\n                            raise ValueError(\'YOLO detection expects 5 columns; segmentation/other format is not inferred\')\n                        class_id = int(fields[0])\n                        if class_id < 0 or names and class_id not in names:\n                            raise ValueError(\'Class ID outside class map\')\n                        cx, cy, width, height = map(float, fields[1:])\n                        raw_boxes.append((str(class_id), names.get(class_id, f\'class_{class_id}\'), bool(names),\n                                          source, cx - width/2, cy - height/2, width, height, 0))\n                    except Exception as error:\n                        bad = True; issue(annotation, f\'Line {line_number}: {error}\')\n            elif kind == \'voc\':\n                tree = ET.fromstring(reader.read(annotation, 8 * 1024**2))\n                if tree.tag != \'annotation\':\n                    raise ValueError(\'Not a VOC annotation\')\n                for dimension in (\'width\', \'height\'):\n                    value = tree.findtext(\'size/\' + dimension)\n                    if value is not None and int(value) != row[dimension]:\n                        raise ValueError(\'VOC annotation size differs from image\')\n                for obj in tree.findall(\'object\'):\n                    label = obj.findtext(\'name\')\n                    if not label:\n                        raise ValueError(\'VOC object without class name\')\n                    xmin, ymin, xmax, ymax = [float(obj.findtext(\'bndbox/\' + key)) for key in (\'xmin\', \'ymin\', \'xmax\', \'ymax\')]\n                    raw_boxes.append((label, label, True, \'voc:names\', xmin/row[\'width\'], ymin/row[\'height\'],\n                                      (xmax-xmin)/row[\'width\'], (ymax-ymin)/row[\'height\'], 0))\n            else:\n                annotation_path, categories, image, objects = annotation\n                for dimension in (\'width\', \'height\'):\n                    if dimension in image and int(image[dimension]) != row[dimension]:\n                        raise ValueError(\'COCO annotation size differs from image\')\n                for obj in objects:\n                    class_id = obj[\'category_id\']\n                    if class_id not in categories:\n                        raise ValueError(\'COCO category missing from categories\')\n                    x, y, width, height = map(float, obj[\'bbox\'])\n                    raw_boxes.append((str(class_id), categories[class_id], True, annotation_path,\n                                      x/row[\'width\'], y/row[\'height\'], width/row[\'width\'], height/row[\'height\'], int(obj.get(\'iscrowd\', 0))))\n            for class_id, label, mapped, schema, x, y, width, height, crowd in raw_boxes:\n                if not all(math.isfinite(v) for v in (x, y, width, height)) or min(width, height) <= 0 or min(x, y) < -1e-6 or max(x+width, y+height) > 1+1e-6:\n                    bad = True; issue(name, \'Invalid bbox: nonfinite, empty, negative, or outside image\'); continue\n                boxes.append({\'dataset\': dataset, \'path\': name, \'split\': row[\'split\'], \'format\': kind,\n                              \'label_namespace\': schema, \'class_id\': class_id, \'class_name\': label, \'class_mapped\': mapped,\n                              \'x_norm\': x, \'y_norm\': y, \'width_norm\': width, \'height_norm\': height,\n                              \'area_fraction\': width * height, \'iscrowd\': crowd,\n                              \'short_side_at_640\': min(width*row[\'width\'], height*row[\'height\']) * 640 / max(row[\'width\'], row[\'height\'])})\n                row[\'boxes\'] += 1\n            row[\'annotation_status\'] = \'invalid_annotation\' if bad else \'annotated\' if row[\'boxes\'] else \'empty_annotation\'\n        except Exception as error:\n            row[\'annotation_status\'] = \'invalid_annotation\'\n            issue(name, f\'{kind}: {type(error).__name__}: {error}\')\n\n    summary = {\'dataset\': dataset, \'root\': str(reader.root), \'images_total\': len(images), \'images_scanned\': len(image_rows),\n               \'partial_scan\': len(selected) < len(images), \'images_hashed\': sum(bool(row[\'sha256\']) for row in image_rows),\n               \'hash_partial\': sum(bool(row[\'sha256\']) for row in image_rows) < len(images),\n               \'verify_decode\': verify_decode, \'annotation_format\': annotation_format,\n               \'valid_boxes\': len(boxes), \'issues\': len(issues),\n               \'invalid_annotations\': sum(row[\'annotation_status\'] == \'invalid_annotation\' for row in image_rows),\n               \'annotation_status\': dict(Counter(row[\'annotation_status\'] for row in image_rows)),\n               \'formats\': dict(Counter(row[\'format\'] for row in image_rows if row[\'format\'])),\n               \'file_extensions\': dict(Counter(PurePosixPath(name).suffix.lower() for name in reader.entries)),\n               \'class_maps\': [{\'path\': name, \'names\': names} for name, names in catalogs]}\n    return {\'summary\': summary, \'images\': image_rows, \'boxes\': boxes, \'issues\': issues}\n\n\ndef duplicate_groups(image_rows):\n    grouped = defaultdict(list)\n    for row in image_rows:\n        if row.get(\'sha256\'):\n            grouped[row[\'sha256\']].append(row)\n    return [{\'sha256\': digest, \'copies\': len(rows), \'cross_dataset\': len({r[\'dataset\'] for r in rows}) > 1,\n             \'cross_split\': len({r[\'split\'] for r in rows} - {\'unknown\'}) > 1,\n             \'members\': [{\'dataset\': r[\'dataset\'], \'path\': r[\'path\'], \'split\': r[\'split\']} for r in rows]}\n            for digest, rows in grouped.items() if len(rows) > 1]\n', 'training/data/vitsd_review.py': '"""Read-only ZIP audit and CAR35 review evidence. Never prepares a training split."""\nfrom collections import Counter, defaultdict\nimport csv\nimport hashlib\nfrom html import escape\nimport json\nfrom pathlib import Path, PurePosixPath\nimport re\nimport shutil\nimport tempfile\nimport zipfile\n\nfrom PIL import Image, ImageDraw, ImageOps\n\nfrom training.data.audit import DatasetReader, audit_dataset\nfrom training.data.common import CAR35, digest_file, write_json\n\n\ndef _validate_zip(path):\n    with zipfile.ZipFile(path) as archive:\n        seen = set()\n        for item in archive.infolist():\n            name = PurePosixPath(item.filename)\n            key = str(name).casefold()\n            if (name.is_absolute() or \'..\' in name.parts or \'\\\\\' in item.filename\n                    or \':\' in item.filename or key in seen or item.flag_bits & 1\n                    or (item.external_attr >> 16) & 0o170000 == 0o120000):\n                raise ValueError(\'Unsafe, encrypted or duplicate ZIP member: \' + item.filename)\n            seen.add(key)\n\n\ndef copy_zip_to_local(source, target, *, progress=None):\n    """Copy the mounted ZIP once, checking its receipt on reuse; no extraction."""\n    source, target = Path(source).resolve(), Path(target).resolve()\n    if source == target:\n        raise ValueError(\'Local staging must differ from the mounted source\')\n    stat = source.stat()\n    identity = dict(source=str(source), size=stat.st_size, mtime_ns=stat.st_mtime_ns)\n    receipt = target.with_suffix(target.suffix + \'.source.json\')\n    if target.exists():\n        if not receipt.is_file():\n            raise FileExistsError(\'Existing local ZIP has no receipt; choose a new target\')\n        saved = json.loads(receipt.read_text(encoding=\'utf-8\'))\n        if saved[\'source_identity\'] != identity or digest_file(target) != saved[\'sha256\']:\n            raise ValueError(\'Source or local ZIP changed; choose a new staging path\')\n        _validate_zip(target)\n        return target\n    target.parent.mkdir(parents=True, exist_ok=True)\n    if shutil.disk_usage(target.parent).free < stat.st_size + 512 * 1024**2:\n        raise OSError(\'Not enough local disk space for the ZIP and report; use another runtime/disk\')\n    _validate_zip(source)\n    partial = target.with_suffix(target.suffix + \'.part\')\n    if partial.exists():\n        raise FileExistsError(\'An interrupted local copy exists; remove only its .part or choose a new target\')\n    digest, copied = hashlib.sha256(), 0\n    try:\n        with source.open(\'rb\') as incoming, partial.open(\'xb\') as outgoing:\n            for block in iter(lambda: incoming.read(8 * 1024**2), b\'\'):\n                outgoing.write(block); digest.update(block); copied += len(block)\n                if progress:\n                    progress(copied)\n        after = source.stat()\n        if copied != stat.st_size or (after.st_size, after.st_mtime_ns) != (stat.st_size, stat.st_mtime_ns):\n            raise ValueError(\'Mounted source changed during copy\')\n        _validate_zip(partial)\n        if target.exists():\n            raise FileExistsError(target)\n        partial.rename(target)\n        write_json(receipt, dict(source_identity=identity, sha256=digest.hexdigest()))\n    finally:\n        if partial.exists():\n            partial.unlink()  # This invocation created this staging file.\n    return target\n\n\ndef audit_zip(source, *, annotation_format=\'yolo\', image_limit=None, cache_dir=None, progress=None):\n    """Fully decode/hash the selected scope. Cache identity includes source and code."""\n    if annotation_format not in (\'yolo\', \'coco\'):\n        raise ValueError(\'Choose yolo or coco explicitly; formats are not merged\')\n    source = Path(source).resolve()\n    _validate_zip(source)\n    from training.data import audit as audit_module\n    implementation = hashlib.sha256((Path(__file__).read_bytes() + Path(audit_module.__file__).read_bytes())).hexdigest()\n    request = dict(source_path=str(source), source_sha256=digest_file(source), annotation_format=annotation_format,\n                   image_limit=image_limit, verify_decode=True, hash_limit=None, implementation=implementation)\n    cache = None\n    if cache_dir is not None:\n        directory = Path(cache_dir); directory.mkdir(parents=True, exist_ok=True)\n        key = hashlib.sha256(json.dumps(request, sort_keys=True).encode()).hexdigest()\n        cache = directory / (key + \'.json\')\n        checksum = cache.with_suffix(\'.json.sha256\')\n        if cache.is_file():\n            if not checksum.is_file() or checksum.read_text().strip() != digest_file(cache):\n                raise ValueError(\'Audit cache changed; choose a fresh cache directory\')\n            saved = json.loads(cache.read_text(encoding=\'utf-8\'))\n            if saved[\'request\'] != request:\n                raise ValueError(\'Audit cache request changed\')\n            return saved[\'report\']\n    with DatasetReader(source) as reader:\n        if reader.warnings:\n            raise ValueError(\'Unsupported archive contents: \' + str(reader.warnings[:3]))\n        report = audit_dataset(reader, \'vitsd\', annotation_format=annotation_format,\n                               image_limit=image_limit, hash_limit=None, verify_decode=True, progress=progress)\n    report[\'summary\'][\'source_sha256\'] = request[\'source_sha256\']\n    report[\'summary\'][\'audit_implementation_sha256\'] = implementation\n    # JSON object keys (class IDs) must have the same representation before and\n    # after cache restore, not int on the first run and str on the second.\n    report = json.loads(json.dumps(report, ensure_ascii=False, allow_nan=False))\n    if cache is not None:\n        with cache.open(\'x\', encoding=\'utf-8\') as stream:\n            json.dump(dict(request=request, report=report), stream, ensure_ascii=False, allow_nan=False)\n        checksum.write_text(digest_file(cache), encoding=\'utf-8\')\n    return report\n\n\ndef load_reference(path):\n    """Read only CAR35 manifest metadata, from JSON or a ZIP containing one manifest."""\n    if not path:\n        return None\n    path = Path(path)\n    if path.suffix.lower() == \'.zip\':\n        _validate_zip(path)\n        with zipfile.ZipFile(path) as archive:\n            matches = [n for n in archive.namelist() if PurePosixPath(n).name == \'manifest.json\']\n            if len(matches) != 1:\n                raise ValueError(\'Reference ZIP must contain exactly one manifest.json\')\n            if archive.getinfo(matches[0]).file_size > 128 * 1024**2:\n                raise ValueError(\'Reference manifest exceeds read limit\')\n            manifest = json.loads(archive.read(matches[0]).decode(\'utf-8-sig\'))\n    else:\n        if path.stat().st_size > 128 * 1024**2:\n            raise ValueError(\'Reference manifest exceeds read limit\')\n        manifest = json.loads(path.read_text(encoding=\'utf-8-sig\'))\n    rows = manifest.get(\'images\')\n    if not isinstance(rows, list) or not rows:\n        raise ValueError(\'Reference manifest must contain image hashes and splits\')\n    hashes = defaultdict(set)\n    for row in rows:\n        digest, split = row.get(\'sha256\', \'\'), row.get(\'split\')\n        if not re.fullmatch(\'[a-f0-9]{64}\', digest) or split not in (\'train\', \'val\', \'test\'):\n            raise ValueError(\'Invalid reference SHA-256 or split\')\n        hashes[digest].add(split)\n    return {key: sorted(values) for key, values in hashes.items()}\n\n\ndef build_review(report, *, reference=None):\n    by_image, hashes = defaultdict(list), defaultdict(list)\n    for box in report[\'boxes\']:\n        by_image[box[\'path\']].append(box)\n    for row in report[\'images\']:\n        if row[\'sha256\']:\n            hashes[row[\'sha256\']].append(row)\n    duplicates, conflicting = [], set()\n    for digest, rows in sorted(hashes.items()):\n        if len(rows) < 2:\n            continue\n        signatures = set()\n        for row in rows:\n            signature = tuple(sorted((b[\'class_name\'], b[\'class_mapped\'], b[\'iscrowd\'],\n                                      *(round(b[k], 8) for k in (\'x_norm\', \'y_norm\', \'width_norm\', \'height_norm\')))\n                                     for b in by_image[row[\'path\']]))\n            signatures.add((row[\'annotation_status\'], signature))\n        conflict = len(signatures) > 1\n        if conflict:\n            conflicting.add(digest)\n        duplicates.append(dict(sha256=digest, copies=len(rows), label_conflict=conflict,\n                               cross_split=len({r[\'split\'] for r in rows} - {\'unknown\'}) > 1,\n                               paths=[r[\'path\'] for r in rows]))\n    catalog = {}\n    for entry in report[\'summary\'][\'class_maps\']:\n        for key, name in entry[\'names\'].items():\n            catalog[entry[\'path\'], str(key)] = name\n    for b in report[\'boxes\']:\n        catalog[b[\'label_namespace\'], str(b[\'class_id\'])] = b[\'class_name\']\n    mapping = [dict(namespace=namespace, source_id=key, source_name=name,\n                    candidate_target_id=CAR35.index(name) if name in CAR35 else \'\',\n                    candidate_target_name=name if name in CAR35 else \'\', approved=False,\n                    action=\'review_exact_name\' if name in CAR35 else \'review_exclude_or_map\')\n               for (namespace, key), name in sorted(catalog.items())]\n    decisions, overlap = [], []\n    for row in report[\'images\']:\n        boxes = by_image[row[\'path\']]\n        reasons = []\n        if row[\'annotation_status\'] not in (\'annotated\', \'empty_annotation\'):\n            reasons.append(row[\'annotation_status\'])\n        if any(not b[\'class_mapped\'] or b[\'iscrowd\'] for b in boxes):\n            reasons.append(\'unmapped_or_crowd_annotation\')\n        if row[\'sha256\'] in conflicting:\n            reasons.append(\'conflicting_duplicate_annotations\')\n        selected = [b for b in boxes if b[\'class_mapped\'] and b[\'class_name\'] in CAR35]\n        decision = \'quarantine\' if reasons else \'review_car35\' if selected else \'review_background\'\n        decisions.append(dict(path=row[\'path\'], split=row[\'split\'], sha256=row[\'sha256\'],\n                              annotation_status=row[\'annotation_status\'], decision=decision,\n                              reason=\';\'.join(reasons), car35_boxes=len(selected)))\n        if reference is not None and row[\'sha256\'] in reference:\n            overlap.append(dict(path=row[\'path\'], split=row[\'split\'], sha256=row[\'sha256\'],\n                                reference_splits=\';\'.join(reference[row[\'sha256\']])))\n    summary = dict(source=report[\'summary\'], training_ready=False, mapping_approved=False,\n                   independent_field_test=False, unique_hashed_images=len(hashes),\n                   duplicate_groups=len(duplicates), conflicting_duplicate_groups=len(conflicting),\n                   decisions=dict(Counter(r[\'decision\'] for r in decisions)),\n                   reference_exact_check=\'not_supplied\' if reference is None else \'checked\',\n                   reference_images=None if reference is None else len(reference),\n                   reference_inventory_sha256=None if reference is None else hashlib.sha256(\n                       json.dumps(reference, sort_keys=True).encode()).hexdigest(),\n                   reference_overlap_images=len(overlap) if reference is not None else None,\n                   reference_near_check=\'not_checked\',\n                   limitations=\'Review candidates only. No images, labels or splits changed. \'\n                               \'Exact-name mapping is unapproved. dHash is a heuristic, not capture metadata. \'\n                               \'A hash manifest cannot establish cross-source scene independence.\')\n    return dict(summary=summary, mapping=mapping, decisions=decisions, duplicates=duplicates,\n                reference_overlap=overlap)\n\n\ndef _csv(path, rows, fields):\n    with path.open(\'w\', newline=\'\', encoding=\'utf-8-sig\') as stream:\n        writer = csv.DictWriter(stream, fieldnames=fields)\n        writer.writeheader(); writer.writerows(rows)\n\n\ndef _near_edges(reader, rows, threshold, progress):\n    """BK-tree spanning forest: bounded by images, never an all-pairs matrix."""\n    parent = list(range(len(rows)))\n    def find(i):\n        while parent[i] != i:\n            parent[i] = parent[parent[i]]; i = parent[i]\n        return i\n    tree, edges = None, []\n    values = progress(list(enumerate(rows)), desc=\'Visual hashes / candidate groups\', unit=\'image\') if progress else enumerate(rows)\n    for i, row in values:\n        if row[\'annotation_status\'] == \'unreadable_image\':\n            continue\n        with reader.open(row[\'path\']) as stream, Image.open(stream) as image:\n            pixels = list(image.convert(\'L\').resize((9, 8), Image.Resampling.LANCZOS).tobytes())\n        value = 0\n        for y in range(8):\n            for x in range(8):\n                value = (value << 1) | (pixels[y*9+x] > pixels[y*9+x+1])\n        if tree is None:\n            tree = (value, i, {}); continue\n        pending = [tree]\n        while pending:\n            other, j, children = pending.pop()\n            distance = (value ^ other).bit_count()\n            a, b = find(i), find(j)\n            if distance <= threshold and a != b:\n                parent[max(a, b)] = min(a, b)\n                edges.append(dict(left=row[\'path\'], right=rows[j][\'path\'], distance=distance,\n                                  same_sha256=row[\'sha256\'] == rows[j][\'sha256\'],\n                                  cross_split=row[\'split\'] != rows[j][\'split\']))\n            pending.extend(child for edge, child in children.items()\n                           if distance-threshold <= edge <= distance+threshold)\n        node = tree\n        while True:\n            distance = (value ^ node[0]).bit_count()\n            if distance == 0:\n                break\n            if distance not in node[2]:\n                node[2][distance] = (value, i, {}); break\n            node = node[2][distance]\n    sizes = Counter(find(i) for i in range(len(rows)) if rows[i][\'sha256\'])\n    return edges, dict(candidate_groups=len(sizes), largest_candidate_group=max(sizes.values(), default=0),\n                       joining_edges=len(edges), cross_split_edges=sum(e[\'cross_split\'] for e in edges),\n                       threshold=threshold, all_pairs=False, split_changed=False)\n\n\ndef _sheet(reader, rows, boxes, target, title):\n    width, height, columns = 360, 240, 3\n    sheet = Image.new(\'RGB\', (width*columns, height*((len(rows)+columns-1)//columns)), \'#18212b\')\n    canvas = ImageDraw.Draw(sheet)\n    for i, row in enumerate(rows):\n        x, y = (i % columns)*width, (i // columns)*height\n        with reader.open(row[\'path\']) as stream, Image.open(stream) as source:\n            image = source.convert(\'RGB\')\n        draw = ImageDraw.Draw(image)\n        for b in boxes[row[\'path\']]:\n            left, top = b[\'x_norm\']*image.width, b[\'y_norm\']*image.height\n            draw.rectangle((left, top, left+b[\'width_norm\']*image.width, top+b[\'height_norm\']*image.height),\n                           outline=\'yellow\', width=2)\n            draw.text((max(0, left), max(0, top-12)), b[\'class_name\'].encode(\'ascii\', \'replace\').decode(), fill=\'yellow\')\n        image = ImageOps.contain(image, (width-8, height-48))\n        sheet.paste(image, (x+(width-image.width)//2, y+4))\n        caption = f"{row[\'split\']} | {PurePosixPath(row[\'path\']).name}"\n        canvas.text((x+4, y+height-40), title[:50], fill=\'white\')\n        canvas.text((x+4, y+height-24), caption[:52], fill=\'white\')\n    sheet.save(target, quality=85)\n\n\ndef write_review(source, report, review, output, *, near_threshold=4, progress=None):\n    """Publish a new, portable report with bounded previews; no training output."""\n    source, output = Path(source).resolve(), Path(output).resolve()\n    if output.exists():\n        raise FileExistsError(output)\n    if near_threshold is not None and (type(near_threshold) is not int or not 0 <= near_threshold <= 8):\n        raise ValueError(\'near_threshold must be None or an integer from 0 to 8\')\n    if digest_file(source) != report[\'summary\'][\'source_sha256\']:\n        raise ValueError(\'Source ZIP changed after audit\')\n    output.parent.mkdir(parents=True, exist_ok=True)\n    with tempfile.TemporaryDirectory(prefix=\'.vitsd-review-\', dir=output.parent) as temporary, DatasetReader(source) as reader:\n        stage = Path(temporary)/\'report\'; stage.mkdir(); (stage/\'previews\').mkdir()\n        summary = dict(review[\'summary\'])\n        write_json(stage/\'source_summary.json\', report[\'summary\'])\n        write_json(stage/\'exact_duplicates.json\', review[\'duplicates\'])\n        for name, rows, fields in [\n            (\'images\', report[\'images\'], list(report[\'images\'][0]) if report[\'images\'] else [\'path\']),\n            (\'boxes\', report[\'boxes\'], list(report[\'boxes\'][0]) if report[\'boxes\'] else [\'path\', \'class_name\']),\n            (\'issues\', report[\'issues\'], [\'dataset\', \'path\', \'message\']),\n            (\'mapping_candidates\', review[\'mapping\'], [\'namespace\', \'source_id\', \'source_name\', \'candidate_target_id\', \'candidate_target_name\', \'approved\', \'action\']),\n            (\'decisions\', review[\'decisions\'], [\'path\', \'split\', \'sha256\', \'annotation_status\', \'decision\', \'reason\', \'car35_boxes\']),\n            (\'reference_overlap\', review[\'reference_overlap\'], [\'path\', \'split\', \'sha256\', \'reference_splits\']),\n        ]:\n            _csv(stage/(name+\'.csv\'), rows, fields)\n        _csv(stage/\'quarantine.csv\', [r for r in review[\'decisions\'] if r[\'decision\']==\'quarantine\'],\n             [\'path\', \'split\', \'sha256\', \'annotation_status\', \'decision\', \'reason\', \'car35_boxes\'])\n        boxes, image_map = defaultdict(list), {r[\'path\']: r for r in report[\'images\']}\n        for b in report[\'boxes\']:\n            boxes[b[\'path\']].append(b)\n        counts = Counter((b[\'class_name\'], b[\'split\']) for b in report[\'boxes\'])\n        class_rows = [dict(name=name, **{s: counts[name, s] for s in (\'train\', \'val\', \'test\', \'unknown\')})\n                      for name in sorted({b[\'class_name\'] for b in report[\'boxes\']})]\n        _csv(stage/\'class_counts.csv\', class_rows, [\'name\', \'train\', \'val\', \'test\', \'unknown\'])\n        summary[\'class_counts_basis\'] = \'Geometrically valid source boxes, including valid boxes in images with other annotation errors; not cleaned training counts.\'\n        summary[\'near_duplicates\'] = dict(status=\'disabled\')\n        edges = []\n        if near_threshold is not None:\n            edges, info = _near_edges(reader, report[\'images\'], near_threshold, progress)\n            summary[\'near_duplicates\'] = dict(status=\'review_candidates\', **info)\n        _csv(stage/\'near_duplicate_edges.csv\', edges, [\'left\', \'right\', \'distance\', \'same_sha256\', \'cross_split\'])\n        previews = []\n        for cls, name in enumerate(CAR35):\n            paths = sorted({b[\'path\'] for b in report[\'boxes\'] if b[\'class_mapped\'] and b[\'class_name\'] == name})[:6]\n            if paths:\n                filename = f\'previews/car35_{cls:02d}.jpg\'\n                _sheet(reader, [image_map[p] for p in paths], boxes, stage/filename, name)\n                previews.append((name, filename))\n        samples = [image_map[r[\'path\']] for r in review[\'decisions\']\n                   if r[\'decision\']==\'quarantine\' and image_map[r[\'path\']][\'annotation_status\']!=\'unreadable_image\'][:6]\n        if samples:\n            _sheet(reader, samples, boxes, stage/\'previews/quarantine.jpg\', \'QUARANTINE - inspect source labels\')\n            previews.append((\'Quarantine examples\', \'previews/quarantine.jpg\'))\n        if edges:\n            # Diverse distances; at most six pairs, one bounded sheet.\n            chosen, seen_distances = [], set()\n            for edge in edges:\n                if edge[\'distance\'] not in seen_distances:\n                    chosen.append(edge); seen_distances.add(edge[\'distance\'])\n                if len(chosen) == 6:\n                    break\n            _sheet(reader, [image_map[e[k]] for e in chosen for k in (\'left\', \'right\')], boxes,\n                   stage/\'previews/near_pairs.jpg\', \'Near candidates: consecutive left/right tiles\')\n            previews.append((\'Near-duplicate pair samples (consecutive tiles)\', \'previews/near_pairs.jpg\'))\n        write_json(stage/\'summary.json\', summary)\n        links = \'\'.join(f\'<li><a href="{p.name}">{p.name}</a></li>\' for p in sorted(stage.iterdir()) if p.is_file())\n        cards = \'\'.join(f\'<h2>{escape(name)}</h2><img src="{escape(path)}" alt="{escape(name)}">\' for name, path in previews)\n        page = (\'<!doctype html><meta charset="utf-8"><meta name="viewport" content="width=device-width,initial-scale=1">\'\n                \'<title>ViTSD CAR35 cleaning review</title><style>body{font:16px/1.5 system-ui;max-width:1150px;margin:auto;padding:24px}\'\n                \'img{max-width:100%;height:auto}pre{white-space:pre-wrap;overflow-wrap:anywhere}a{color:#1763a6}</style>\'\n                \'<h1>ViTSD → CAR35: cleaning review</h1><p>No source edits, split or training dataset produced. \'\n                \'Mapping and dHash links need human review. Counts include source defects; read quarantine.csv. \'\n                \'Previews are bounded examples, not representative samples. Reference hashes check exact overlap only.</p>\'\n                \'<h2>Summary</h2><pre>\'+escape(json.dumps(summary, ensure_ascii=False, indent=2))+\'</pre>\'\n                \'<h2>Files</h2><ul>\'+links+\'</ul>\'+cards)\n        (stage/\'index.html\').write_text(page, encoding=\'utf-8\')\n        stage.rename(output)\n    return summary\n'}
PIPELINE_ID = hashlib.sha256(json.dumps(TRAINING_FILES, sort_keys=True).encode()).hexdigest()
CODE_ROOT = WORK / 'pipeline' / PIPELINE_ID
for relative, source in TRAINING_FILES.items():
    target = CODE_ROOT / relative
    target.parent.mkdir(parents=True, exist_ok=True)
    target.write_text(source, encoding='utf-8')
# Tránh dùng helper cũ còn trong bộ nhớ nếu upload notebook mới cùng runtime.
for name in list(sys.modules):
    if name == 'training' or name.startswith('training.'):
        del sys.modules[name]
sys.path.insert(0, str(CODE_ROOT))
importlib.invalidate_caches()
from training.data.vitsd_review import (copy_zip_to_local, audit_zip, load_reference,
    build_review, write_review)
print('Pipeline:', PIPELINE_ID[:16])


## 2. Chép ZIP một lần về ổ Colab

Copy local tránh hàng chục nghìn lượt đọc nhỏ qua Drive mount. Cần chỗ cho ZIP + khoảng trống làm báo cáo,
không cần thêm một bản toàn bộ dataset giải nén. Runtime mới phải copy lại.
Chạy lại cùng runtime sẽ kiểm tra checksum bản local trước khi dùng lại.


In [ ]:
from tqdm.auto import tqdm
assert DRIVE_ZIP_PATH.strip(), 'Điền DRIVE_ZIP_PATH bằng Copy path của file ZIP trong Drive.'
mounted_zip = Path(DRIVE_ZIP_PATH).expanduser()
assert mounted_zip.is_file(), f'Không thấy ZIP: {mounted_zip}; kiểm tra mount và Copy path.'
assert mounted_zip.suffix.lower() == '.zip', 'Chọn file ZIP, không phải folder.'
if COPY_ZIP_TO_LOCAL:
    # Mỗi đường dẫn nguồn có thư mục local riêng; receipt kiểm tra size/mtime và checksum.
    source_id = hashlib.sha256(str(mounted_zip.resolve()).encode()).hexdigest()[:16]
    with tqdm(total=mounted_zip.stat().st_size, unit='B', unit_scale=True, desc='Copy ZIP Drive → local') as bar:
        def copied(total):
            bar.update(total-bar.n)
        SOURCE_ZIP = copy_zip_to_local(mounted_zip, WORK/'downloads'/source_id/'dataset.zip', progress=copied)
        bar.update(bar.total-bar.n)
else:
    SOURCE_ZIP = mounted_zip
    print('Đọc trực tiếp qua mount có thể chậm. COPY_ZIP_TO_LOCAL=True được khuyến nghị.')
print('Đọc ZIP:', SOURCE_ZIP)


## 3. Audit ảnh/nhãn và đề xuất mapping

Lượt đầu hash ZIP rồi decode/hash từng ảnh, có thể mất nhiều phút. Thanh tiến độ hiện sau bước hash ZIP.
Không tắt cell chỉ vì chưa có GPU usage: bước này dùng CPU/ổ đĩa.
Cache audit có fingerprint nguồn + code + cấu hình; chạy lại cùng runtime không cần audit lại ảnh nếu khớp.
Ảnh thiếu nhãn, bbox lỗi, ảnh hỏng và duplicate có annotation mâu thuẫn được liệt kê để cách ly.
Không tự coi ảnh thiếu nhãn là ảnh nền, không tự sửa bbox về biên ảnh.


In [ ]:
assert 'audit_zip' in globals(), 'Chạy cell pipeline trước.'
print('Đang hash ZIP / kiểm tra cache audit...', flush=True)
audit = audit_zip(SOURCE_ZIP, annotation_format=ANNOTATION_FORMAT, image_limit=MAX_IMAGES,
    cache_dir=WORK/'audit_cache', progress=lambda items, desc, unit: tqdm(items, desc=desc, unit=unit))
reference = load_reference(REFERENCE_MANIFEST_PATH.strip() or None)
review = build_review(audit, reference=reference)
summary = review['summary']
print(f"Audit: {audit['summary']['images_scanned']:,}/{audit['summary']['images_total']:,} ảnh | {len(audit['boxes']):,} bbox có hình học hợp lệ")
print('Partial scan:', audit['summary']['partial_scan'])
print('Trạng thái nhãn:', audit['summary']['annotation_status'])
print('Nhóm trùng byte:', summary['duplicate_groups'], '| Nhóm nhãn mâu thuẫn:', summary['conflicting_duplicate_groups'])
print('Đề xuất xử lý:', summary['decisions'])
print('Đối chiếu CAR35 exact:', summary['reference_exact_check'], '| Ảnh trùng:', summary['reference_overlap_images'])
print('Chưa tạo dataset train; mapping chưa duyệt.')


## 4. Báo cáo có ảnh để duyệt

Xuất mọi dòng CSV; màn hình chỉ in tóm tắt. Ảnh minh họa tối đa 6 ảnh/lớp CAR35 và một số ca cách ly/cặp gần trùng.
dHash tái đọc ảnh và tạo một **forest liên kết ứng viên**, không phải mọi cặp giống nhau; không tự gom/chia dataset train.
Nhóm lớn cần rà bằng mắt trước khi chốt. Kết quả có đầy đủ đường dẫn để tìm lại ảnh/nhãn gốc trong ZIP.
`class_counts.csv` đếm bbox nguồn hợp lệ về hình học, có thể gồm box trong ảnh có lỗi khác; không phải số mẫu sạch sau loại trùng.


In [ ]:
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPORT_ROOT = WORK/'reports'/('vitsd_car35_review_'+RUN_ID)
report_summary = write_review(SOURCE_ZIP, audit, review, REPORT_ROOT,
    near_threshold=NEAR_HASH_DISTANCE,
    progress=lambda items, desc, unit: tqdm(items, desc=desc, unit=unit))
print('Báo cáo:', REPORT_ROOT)
print('dHash:', report_summary['near_duplicates'])
print('Tải ZIP cuối notebook rồi giải nén, mở index.html để xem ảnh và liên kết CSV.')


In [ ]:
import matplotlib.pyplot as plt
from collections import Counter
from training.data.common import CAR35
counts = Counter(b['class_name'] for b in audit['boxes'] if b['class_mapped'])
fig, axes = plt.subplots(1, 2, figsize=(17, 10))
axes[0].barh(range(35), [counts[n] for n in CAR35])
axes[0].set_yticks(range(35), CAR35, fontsize=8)
axes[0].set_title('CAR35 source boxes (before cleaning/dedup)')
axes[1].hist([b['short_side_at_640'] for b in audit['boxes']], bins=45)
axes[1].axvline(16, color='red', linestyle='--')
axes[1].set(xlabel='BBox short side at input 640 (px)', title='Source box sizes')
fig.tight_layout(); fig.savefig(REPORT_ROOT/'distributions.png', dpi=140)
plt.show(); plt.close(fig)


## 5. Lưu báo cáo — không chép ZIP dataset lên Drive lần nữa

Chỉ lưu ZIP **báo cáo nhỏ** vào `MyDrive/SignSight_Reports` và tải về máy nếu bật.
Gửi ZIP báo cáo này để rà mapping, nhãn mâu thuẫn và overlap trước khi tạo dataset V5.
Không dùng ZIP báo cáo làm dataset train. File gốc và split cũ được giữ nguyên.
Nếu runtime bị ngắt, cache local mất; Drive chỉ giữ các báo cáo đã xuất thành công.


In [ ]:
from google.colab import files
RESULT_ZIP = Path(shutil.make_archive(str(WORK/REPORT_ROOT.name), 'zip', REPORT_ROOT))
print(f'Report: {RESULT_ZIP.name} ({RESULT_ZIP.stat().st_size/1024**2:.1f} MiB)')
if SAVE_REPORT_TO_DRIVE:
    DRIVE_REPORT_DIR.mkdir(parents=True, exist_ok=True)
    destination = DRIVE_REPORT_DIR/RESULT_ZIP.name
    with RESULT_ZIP.open('rb') as incoming, destination.open('xb') as outgoing:
        shutil.copyfileobj(incoming, outgoing)
    print('Đã lưu báo cáo trên Drive:', destination)
if DOWNLOAD_REPORT:
    files.download(str(RESULT_ZIP))
